# Handwritten Digit Recognition with a CNN (PyTorch)

A small **convolutional neural network** that recognizes handwritten digits (0-9)
using the 8x8 digits dataset bundled with scikit-learn (no download needed).

**Pipeline:** load images → split → build CNN → train → evaluate → inspect mistakes → save → predict

In [ ]:
# Uncomment if packages are missing (Google Colab has them preinstalled)
# !pip install torch scikit-learn matplotlib numpy

## 1. Imports & config

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

SEED = 42
EPOCHS = 30
BATCH_SIZE = 64
LR = 1e-3

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Load and look at the data

In [ ]:
digits = load_digits()
X, y = digits.images, digits.target  # X shape: (1797, 8, 8), pixel values 0-16
print("Images:", X.shape, "| Classes:", np.unique(y))

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, img, label in zip(axes.ravel(), X[:16], y[:16]):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(f"label: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Split & prepare tensors

In [ ]:
X = (X / 16.0).astype(np.float32)           # scale pixels to 0-1
X = X[:, None, :, :]                        # add channel dim -> (N, 1, 8, 8)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

def to_loader(X_, y_, shuffle):
    ds = TensorDataset(torch.tensor(X_), torch.tensor(y_, dtype=torch.long))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, True)
test_loader = to_loader(X_test, y_test, False)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## 4. The CNN

In [ ]:
class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),   # 8x8
            nn.MaxPool2d(2),                                         # 4x4
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),  # 4x4
            nn.MaxPool2d(2),                                         # 2x2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(32 * 2 * 2, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = DigitCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## 5. Train

In [ ]:
def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(xb)
            correct += (logits.argmax(1) == yb).sum().item()
            n += len(xb)
    return total_loss / n, correct / n

history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, True)
    te_loss, te_acc = run_epoch(test_loader, False)
    for k, v in zip(history, (tr_loss, te_loss, tr_acc, te_acc)):
        history[k].append(v)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | test loss {te_loss:.4f} acc {te_acc:.3f}")

## 6. Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["test_loss"], label="test")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["test_acc"], label="test")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

## 7. Evaluate

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(X_test).to(device))
    preds = logits.argmax(1).cpu().numpy()

print(classification_report(y_test, preds))
ConfusionMatrixDisplay.from_predictions(y_test, preds, cmap="Blues")
plt.show()

## 8. Look at the mistakes

In [ ]:
wrong = np.where(preds != y_test)[0]
print(f"{len(wrong)} wrong out of {len(y_test)}")

if len(wrong):
    n = min(8, len(wrong))
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.6))
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, wrong[:n]):
        ax.imshow(X_test[i, 0], cmap="gray_r")
        ax.set_title(f"true {y_test[i]}\npred {preds[i]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 9. Save and reload

In [ ]:
torch.save(model.state_dict(), "digit_cnn.pt")

loaded = DigitCNN()
loaded.load_state_dict(torch.load("digit_cnn.pt", map_location="cpu"))
loaded.eval()
print("Saved digit_cnn.pt and reloaded it")

## 10. Predict a single image

In [ ]:
idx = 0
sample = torch.tensor(X_test[idx:idx + 1])
with torch.no_grad():
    probs = torch.softmax(loaded(sample), dim=1)[0]

plt.imshow(X_test[idx, 0], cmap="gray_r"); plt.axis("off")
plt.title(f"predicted {probs.argmax().item()} (confidence {probs.max():.1%}) | actual {y_test[idx]}")
plt.show()